In [0]:
-- =============================================================================
-- NOTEBOOK: 04_build_semantic_layer
-- PURPOSE:  Build change-resilient contract views for Power BI
--           This is the ONLY layer Power BI connects to.
--           When source tables change, only these views need updating.
--           Zero dashboard changes required.
-- CATALOG:  people_analytics
-- SCHEMA:   semantic
-- AUTHOR:   Portfolio Project — Adobe People Analytics
-- DATE:     2026-04-09
--
-- ARCHITECTURE PRINCIPLE:
--   Power BI → semantic.* views → gold.* tables → silver.* tables → bronze.*
--   Source changes are absorbed here. Dashboards never break.
--
-- CHANGE MANAGEMENT RULE:
--   Column names and data types in semantic.* views are CONTRACTS.
--   They never change without a migration notice to all dashboard owners.
--   New columns may be added. Existing columns are never renamed or removed.
-- =============================================================================

-- CELL 1 — Set catalog and schema
USE CATALOG people_analytics;
USE SCHEMA semantic;

SELECT 'Semantic layer build started' AS status,
       current_timestamp()            AS run_ts;

In [0]:
-- =============================================================================
-- CELL 2 - semantic.dim_employee_v
-- The primary employee dimension for Power BI
-- Surfaces SCD2 history from silver.dim_employee
--
-- Power BI usage:
--   Current headcount  → filter WHERE is_current = 1 AND status = 'Active'
--   Historical reports → join fact tables on employee_sk
--   SCD2 demo page     → show all versions per emp_id
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.dim_employee_v
COMMENT 'Contract view: employee dimension with SCD2 history.
Power BI connects HERE, never to silver.dim_employee directly.
Column contract: names and types never change without migration notice.
is_current=1: current employee record. is_current=0: historical version.'
AS
SELECT
    employee_sk,
    emp_id,
    version,
    full_name,
    gender,
    ethnicity,
    birth_date,
    employment_type,
    hire_date,
    dept_id,
    dept_name,
    business_unit,
    location,
    region,
    job_code,
    job_title,
    job_family,
    job_level,
    is_people_manager,
    salary,
    comp_band,
    compa_ratio,
    manager_id,
    status,
    termination_date,
    termination_reason,
    termination_type,
    scd_start_date,
    scd_end_date,
    is_current,

    -- Derived columns for Power BI convenience
    CASE
        WHEN is_current = 1 AND status = 'Active' THEN 1
        ELSE 0
    END                                         AS is_active_current,

    DATEDIFF(CURRENT_DATE(), hire_date)         AS tenure_days_today,

    ROUND(DATEDIFF(CURRENT_DATE(), hire_date)
          / 365.25, 2)                          AS tenure_years_today,

    CASE
        WHEN DATEDIFF(CURRENT_DATE(), hire_date) / 365.25 < 1   THEN '< 1 Year'
        WHEN DATEDIFF(CURRENT_DATE(), hire_date) / 365.25 < 3   THEN '1-3 Years'
        WHEN DATEDIFF(CURRENT_DATE(), hire_date) / 365.25 < 5   THEN '3-5 Years'
        WHEN DATEDIFF(CURRENT_DATE(), hire_date) / 365.25 < 10  THEN '5-10 Years'
        ELSE '10+ Years'
    END                                         AS tenure_band_today,

    load_ts
FROM people_analytics.silver.dim_employee;

SELECT 'dim_employee_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.dim_employee_v;

In [0]:
-- =============================================================================
-- CELL 3 - semantic.dim_department_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.dim_department_v
COMMENT 'Contract view: department dimension.
One row per department. Stable reference data.'
AS
SELECT
    dept_id,
    dept_name,
    business_unit,
    cost_centre,
    location,
    region,
    load_ts
FROM people_analytics.silver.dim_department;

SELECT 'dim_department_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.dim_department_v;

In [0]:
-- =============================================================================
-- CELL 4 semantic.dim_job_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.dim_job_v
COMMENT 'Contract view: job dimension.
One row per job code. Includes level sort for correct ordering in Power BI.'
AS
SELECT
    job_code,
    job_title,
    job_family,
    job_level,
    job_level_sort,
    job_level_category,
    is_people_manager,
    load_ts
FROM people_analytics.silver.dim_job;

SELECT 'dim_job_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.dim_job_v;

In [0]:
-- =============================================================================
-- CELL 5 semantic.dim_comp_band_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.dim_comp_band_v
COMMENT 'Contract view: compensation band reference.
Salary ranges defined by Total Rewards team.'
AS
SELECT
    band_name,
    band_min,
    band_mid,
    band_max,
    band_range_pct,
    load_ts,
    CASE band_name
        WHEN 'Band 1 (IC1)' THEN 1
        WHEN 'Band 2 (IC2)' THEN 2
        WHEN 'Band 3 (IC3)' THEN 3
        WHEN 'Band 4 (IC4)' THEN 4
        WHEN 'Band 5 (IC5)' THEN 5
        WHEN 'Band 6 (M1)' THEN 6
        WHEN 'Band 7 (M2+)' THEN 7
    END AS band_sort
FROM people_analytics.silver.dim_comp_band;

SELECT 'dim_comp_band_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.dim_comp_band_v;

In [0]:
CREATE OR REPLACE VIEW people_analytics.semantic.dim_date_v
COMMENT 'Contract view: date dimension with Adobe fiscal year (starts December).
Dynamically filtered to last 5 fiscal years through today.
Use date_sk to join to fact tables. Use date for slicer and axis.'
AS
SELECT
    date_sk,
    date,
    year,
    quarter,
    month,
    month_name,
    month_short,
    week,
    day,
    day_of_week,
    day_name,
    is_weekday,
    is_month_end,
    is_quarter_end,
    fiscal_year,
    fiscal_year * 10 + fiscal_quarter AS fiscal_yyyyq,
    fiscal_year * 1000 + fiscal_quarter * 100 + fiscal_month AS fiscal_yyyyqmm,
    CASE
        WHEN MONTH(date) >= 12
        THEN 'FY' || CAST(YEAR(date) + 1 AS STRING)
        ELSE 'FY' || CAST(YEAR(date) AS STRING)
    END AS fiscal_year_label,
    fiscal_quarter,
    CASE
    WHEN MONTH(date) IN (12, 1, 2) THEN 'Q1'
    WHEN MONTH(date) IN (3, 4, 5)  THEN 'Q2'
    WHEN MONTH(date) IN (6, 7, 8)  THEN 'Q3'
    WHEN MONTH(date) IN (9, 10, 11) THEN 'Q4'
END AS fiscal_quarter_label,
    CONCAT(
        CASE
            WHEN MONTH(date) IN (12, 1, 2) THEN 'Q1'
            WHEN MONTH(date) IN (3, 4, 5)  THEN 'Q2'
            WHEN MONTH(date) IN (6, 7, 8)  THEN 'Q3'
            WHEN MONTH(date) IN (9, 10, 11) THEN 'Q4'
        END,
        ' ',
        CASE
            WHEN MONTH(date) >= 12
            THEN 'FY' || CAST(YEAR(date) + 1 AS STRING)
            ELSE 'FY' || CAST(YEAR(date) AS STRING)
        END
    ) AS fiscal_quarter_year_label,
    fiscal_month,
    yyyymm,
    CONCAT(month_short, ' ', year)  AS month_year_label,
    CONCAT('Q', quarter, ' ', year) AS quarter_year_label
FROM people_analytics.silver.dim_date
WHERE date <= CURRENT_DATE()
  AND date >= CASE
    WHEN MONTH(CURRENT_DATE()) >= 12
    THEN DATE_TRUNC('MONTH', MAKE_DATE(YEAR(CURRENT_DATE()) - 5, 12, 1))
    ELSE DATE_TRUNC('MONTH', MAKE_DATE(YEAR(CURRENT_DATE()) - 6, 12, 1))
  END;

In [0]:
-- =============================================================================
-- CELL 7 semantic.fact_headcount_v
-- Primary fact table for headcount, tenure, and payroll metrics
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.fact_headcount_v
COMMENT 'Contract view: monthly headcount snapshot fact table.
GRAIN: employee × month. SUM(is_active) = headcount at any point.
Fully additive — safe to aggregate across all dimensions.
Use snapshot_yyyymm to join to dim_date_v on yyyymm.'
AS
SELECT
    snapshot_sk,
    date_sk,
    employee_sk,
    dept_id,
    job_code,
    comp_band,
    snapshot_month_start,
    snapshot_month_end,
    snapshot_year,
    snapshot_month,
    snapshot_yyyymm,
    emp_id,
    full_name,
    gender,
    ethnicity,
    employment_type,
    dept_name,
    business_unit,
    region,
    job_title,
    job_family,
    job_level,
    is_people_manager,
    salary,
    compa_ratio,
    status,
    is_active,
    is_new_hire,
    is_termination,
    is_voluntary_exit,
    tenure_days,
    tenure_years,
    tenure_band,
    load_ts
FROM people_analytics.gold.fact_headcount_snapshot;

SELECT 'fact_headcount_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.fact_headcount_v;

In [0]:
-- =============================================================================
-- CELL 8 semantic.fact_attrition_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.fact_attrition_v
COMMENT 'Contract view: attrition event fact table.
GRAIN: one row per employee termination.
is_regrettable=1: high performer voluntary exit — key metric for CHRO.'
AS
SELECT
    attrition_sk,
    date_sk,
    employee_sk,
    emp_id,
    full_name,
    dept_id,
    dept_name,
    business_unit,
    job_level,
    job_family,
    gender,
    ethnicity,
    termination_date,
    termination_reason,
    termination_type,
    exit_year,
    exit_month,
    tenure_at_exit_days,
    tenure_at_exit_years,
    is_90_day_exit,
    was_high_performer,
    is_regrettable,
    salary_at_exit,
    compa_ratio_at_exit,
    load_ts
FROM people_analytics.gold.fact_attrition_event;

SELECT 'fact_attrition_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.fact_attrition_v;

In [0]:
-- =============================================================================
-- CELL  9 semantic.fact_compensation_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.fact_compensation_v
COMMENT 'Contract view: compensation change history.
GRAIN: one row per salary change event.
Use change_reason to filter by Merit / Promotion / Equity Adjustment.'
AS
SELECT
    comp_sk,
    date_sk,
    employee_sk,
    emp_id,
    effective_date,
    effective_year,
    effective_month,
    old_salary,
    new_salary,
    change_amount,
    change_pct,
    change_reason,
    comp_band,
    compa_ratio,
    range_penetration,
    dept_name,
    business_unit,
    job_level,
    job_family,
    gender,
    load_ts
FROM people_analytics.gold.fact_compensation_history;

SELECT 'fact_compensation_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.fact_compensation_v;

In [0]:
-- =============================================================================
-- CELL 10 semantic.fact_performance_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.fact_performance_v
COMMENT 'Contract view: performance review fact table.
GRAIN: one row per employee per review cycle.
Cycles: 2023H1, 2023H2, 2024H1, 2024H2.'
AS
SELECT
    review_sk,
    date_sk,
    employee_sk,
    emp_id,
    review_cycle,
    review_date,
    rating,
    rating_label,
    is_high_performer,
    is_pip,
    dept_name,
    business_unit,
    job_level,
    job_family,
    gender,
    manager_id,
    load_ts
FROM people_analytics.gold.fact_performance_review;

SELECT 'fact_performance_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.fact_performance_v;

In [0]:
-- =============================================================================
-- CELL 11 semantic.v_headcount_trend_v
-- Pre-computed trend for Power BI line charts
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.v_headcount_trend_v
COMMENT 'Contract view: pre-computed monthly headcount trend.
Power BI line charts read columns directly — no DAX time intelligence needed.
MoM, YoY, and rolling averages are all pre-computed at refresh time.'
AS
SELECT
    snapshot_yyyymm,
    snapshot_year,
    snapshot_month,
    snapshot_month_start,
    snapshot_month_end,
    active_headcount,
    new_hires,
    total_exits,
    voluntary_exits,
    net_change,
    avg_salary,
    avg_tenure_years,
    prior_month_hc,
    mom_hc_delta,
    mom_hc_pct,
    prior_year_hc,
    yoy_hc_delta,
    yoy_hc_pct,
    rolling_3m_avg_hc,
    rolling_12m_avg_hc,
    cumulative_joiners,
    load_ts
FROM people_analytics.gold.v_headcount_trend;

SELECT 'v_headcount_trend_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.v_headcount_trend_v;

In [0]:
-- =============================================================================
-- CELL 12 semantic.v_attrition_trend_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.v_attrition_trend_v
COMMENT 'Contract view: pre-computed monthly attrition trend.
Rolling 3M and 12M averages pre-computed — feeds attrition line chart.'
AS
SELECT
    snapshot_yyyymm,
    snapshot_year,
    snapshot_month,
    snapshot_month_start,
    active_hc,
    total_exits,
    voluntary_exits,
    regrettable_exits,
    voluntary_attrition_rate,
    total_attrition_rate,
    regrettable_attrition_rate,
    rolling_3m_attrition,
    rolling_12m_attrition,
    prior_year_attrition_rate,
    load_ts
FROM people_analytics.gold.v_attrition_trend;

SELECT 'v_attrition_trend_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.v_attrition_trend_v;

In [0]:
-- =============================================================================
-- CELL 13 semantic.v_compensation_trend_v
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.v_compensation_trend_v
COMMENT 'Contract view: pre-computed compensation trend by band.
Feeds salary trend line chart in Power BI compensation page.'
AS
SELECT
    snapshot_yyyymm,
    snapshot_year,
    snapshot_month,
    snapshot_month_start,
    comp_band,
    headcount,
    avg_salary,
    avg_compa_ratio,
    min_salary,
    max_salary,
    prior_month_avg_salary,
    mom_salary_delta,
    prior_year_avg_salary,
    yoy_salary_pct,
    load_ts
FROM people_analytics.gold.v_compensation_trend;

SELECT 'v_compensation_trend_v rows:' AS view_name, COUNT(*) AS row_count
FROM people_analytics.semantic.v_compensation_trend_v;

In [0]:

-- =============================================================================
-- CELL 14 dim_month_v
-- =============================================================================
USE CATALOG people_analytics;
USE SCHEMA semantic;

CREATE OR REPLACE VIEW people_analytics.semantic.dim_month_v
COMMENT 'Month-level date dimension for trend views.
One row per month — 39 rows covering Jan 2023 to Apr 2026.
Use this to join trend views (v_headcount_trend_v,
v_attrition_trend_v, v_compensation_trend_v) on yyyymm.
Do NOT use dim_date_v for trend view joins — wrong grain.'
AS
SELECT DISTINCT
    yyyymm,
    MIN(date)                                   AS month_start,
    MAX(date)                                   AS month_end,
    MIN(year)                                   AS year,
    MIN(month)                                  AS month,
    MIN(month_name)                             AS month_name,
    MIN(month_short)                            AS month_short,
    MIN(quarter)                                AS quarter,
    MIN(fiscal_year)                            AS fiscal_year,
    MIN(fiscal_quarter)                         AS fiscal_quarter,
    CONCAT(MIN(month_short), ' ', MIN(year))    AS month_year_label,
    CONCAT('Q', MIN(quarter), ' ', MIN(year))   AS quarter_year_label,
    CONCAT('FY', MIN(fiscal_year))              AS fiscal_year_label
FROM people_analytics.silver.dim_date
GROUP BY yyyymm
ORDER BY yyyymm;

-- Verify
SELECT COUNT(*) AS row_count,
       MIN(yyyymm) AS first_month,
       MAX(yyyymm) AS last_month
FROM people_analytics.semantic.dim_month_v;

In [0]:
-- =============================================================================
-- CELL 15 fact_budget_v
-- PURPOSE: Finance headcount and payroll budget
-- GRAIN: department x month
-- SOURCE: bronze.finance_budget_raw
-- =============================================================================

CREATE OR REPLACE VIEW people_analytics.semantic.fact_budget_v
COMMENT 'Finance headcount and payroll budget.
GRAIN: department x month — one row per dept per month.
approved_headcount = Finance approved HC plan.
approved_payroll = Finance approved payroll budget.
Join to fact_headcount_v on dept_id + snapshot_yyyymm
for actual vs budget variance.
Join to dim_date_v on date_sk for time filtering.
Owner: People Analytics BI team.'
AS
SELECT
    budget_id,
    dept_id,
    dept_name,
    business_unit,
    CAST(budget_month AS DATE)                      AS budget_month,
    CAST(DATE_FORMAT(
        CAST(budget_month AS DATE),
        'yyyyMMdd') AS INT)                         AS date_sk,
    CAST(DATE_FORMAT(
        CAST(budget_month AS DATE),
        'yyyyMM') AS INT)                           AS snapshot_yyyymm,
    CAST(YEAR(CAST(budget_month AS DATE)) AS INT)   AS budget_year,
    CAST(MONTH(CAST(budget_month AS DATE)) AS INT)  AS budget_month_num,
    CAST(approved_headcount AS INT)                 AS approved_headcount,
    CAST(approved_payroll AS DECIMAL(18,2))         AS approved_payroll,
    ROUND(approved_payroll /
          NULLIF(approved_headcount, 0), 2)         AS avg_payroll_per_hc,
    load_ts
FROM people_analytics.bronze.finance_budget_raw
WHERE budget_month IS NOT NULL;

-- Verify
SELECT
    COUNT(*)                    AS row_count,
    COUNT(DISTINCT dept_id)     AS departments,
    MIN(budget_month)           AS first_month,
    MAX(budget_month)           AS last_month,
    SUM(approved_headcount)     AS total_approved_hc
FROM people_analytics.semantic.fact_budget_v;

In [0]:
-- =============================================================================
-- CELL 16 Create Global Filters
-- =============================================================================
-- slicer_gender_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_gender_v
COMMENT 'Gender slicer. No relationship in Power BI.
Use IN VALUES(slicer_gender_v[gender]) in DAX measures.'
AS
SELECT DISTINCT gender
FROM people_analytics.silver.dim_employee
WHERE gender IS NOT NULL
ORDER BY gender;
-- 4 rows

-- slicer_business_unit_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_business_unit_v
COMMENT 'Business unit slicer. No relationship in Power BI.
Use IN VALUES(slicer_business_unit_v[business_unit]) in DAX.'
AS
SELECT DISTINCT business_unit
FROM people_analytics.silver.dim_department
ORDER BY business_unit;
-- 6 rows

-- slicer_dept_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_dept_v
COMMENT 'Department slicer. No relationship in Power BI.
Use IN VALUES(slicer_dept_v[dept_name]) in DAX measures.'
AS
SELECT DISTINCT dept_id, dept_name, business_unit
FROM people_analytics.silver.dim_department
ORDER BY business_unit, dept_name;
-- 18 rows

-- slicer_job_level_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_job_level_v
COMMENT 'Job level slicer. No relationship in Power BI.
job_level_sort ensures IC1→IC2→IC3→IC4→IC5→M1→M2→M3 order.
Use IN VALUES(slicer_job_level_v[job_level]) in DAX measures.'
AS
SELECT DISTINCT
    job_level,
    job_level_sort,
    job_level_category
FROM people_analytics.silver.dim_job
ORDER BY job_level_sort;
-- 8 rows

-- slicer_region_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_region_v
COMMENT 'Region slicer. No relationship in Power BI.
Use IN VALUES(slicer_region_v[region]) in DAX measures.'
AS
SELECT DISTINCT region
FROM people_analytics.silver.dim_department
ORDER BY region;
-- 3 rows

-- slicer_comp_band_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_comp_band_v
COMMENT 'Comp band slicer. No relationship in Power BI.
band_sort ensures correct Band 1 to Band 7 ordering.
Use IN VALUES(slicer_comp_band_v[comp_band]) in DAX measures.'
AS
SELECT
    band_name     AS comp_band,
    CASE band_name
        WHEN 'Band 1 (IC1)' THEN 1
        WHEN 'Band 2 (IC2)' THEN 2
        WHEN 'Band 3 (IC3)' THEN 3
        WHEN 'Band 4 (IC4)' THEN 4
        WHEN 'Band 5 (IC5)' THEN 5
        WHEN 'Band 6 (M1)'  THEN 6
        WHEN 'Band 7 (M2+)' THEN 7
    END           AS band_sort
FROM people_analytics.silver.dim_comp_band
ORDER BY band_sort;
-- 7 rows

-- slicer_tenure_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_tenure_v
COMMENT 'Tenure band slicer. No relationship in Power BI.
tenure_sort ensures correct < 1 Year to 10+ Years ordering.
Use IN VALUES(slicer_tenure_v[tenure_band]) in DAX measures.'
AS
SELECT * FROM (
    VALUES
    ('< 1 Year',   1),
    ('1-3 Years',  2),
    ('3-5 Years',  3),
    ('5-10 Years', 4),
    ('10+ Years',  5)
) AS t(tenure_band, tenure_sort);
-- 5 rows

-- slicer_exit_reason_v
CREATE OR REPLACE VIEW people_analytics.semantic.slicer_exit_reason_v
COMMENT 'Exit reason slicer. No relationship in Power BI.
Use IN VALUES(slicer_exit_reason_v[termination_reason])
in DAX attrition measures.'
AS
SELECT * FROM (
    VALUES
    ('Growth Opportunity',       'Voluntary',    1),
    ('Compensation',             'Voluntary',    2),
    ('Manager/Culture',          'Voluntary',    3),
    ('Life Event',               'Voluntary',    4),
    ('Involuntary - PIP',        'Involuntary',  5),
    ('Involuntary - Restructure','Involuntary',  6),
    ('Retirement',               'Retirement',   7)
) AS t(termination_reason, termination_type, sort_order);
-- 7 rows

In [0]:
-- CELL 17 Check all fact table columns
SELECT 'fact_headcount_v' AS tbl, column_name
FROM people_analytics.information_schema.columns
WHERE table_schema = 'semantic'
  AND table_name = 'fact_headcount_v'
UNION ALL
SELECT 'fact_attrition_v', column_name
FROM people_analytics.information_schema.columns
WHERE table_schema = 'semantic'
  AND table_name = 'fact_attrition_v'
UNION ALL
SELECT 'fact_compensation_v', column_name
FROM people_analytics.information_schema.columns
WHERE table_schema = 'semantic'
  AND table_name = 'fact_compensation_v'
UNION ALL
SELECT 'fact_performance_v', column_name
FROM people_analytics.information_schema.columns
WHERE table_schema = 'semantic'
  AND table_name = 'fact_performance_v'
ORDER BY tbl, column_name;

In [0]:
-- =============================================================================
-- CELL 18 Semantic layer inventory
-- All views deployed and row counts confirmed
-- =============================================================================

SELECT view_name, row_count FROM (
    SELECT 'dim_employee_v'         AS view_name,
           COUNT(*) AS row_count FROM people_analytics.semantic.dim_employee_v
    UNION ALL
    SELECT 'dim_department_v',
           COUNT(*) FROM people_analytics.semantic.dim_department_v
    UNION ALL
    SELECT 'dim_job_v',
           COUNT(*) FROM people_analytics.semantic.dim_job_v
    UNION ALL
    SELECT 'dim_comp_band_v',
           COUNT(*) FROM people_analytics.semantic.dim_comp_band_v
    UNION ALL
    SELECT 'dim_date_v',
           COUNT(*) FROM people_analytics.semantic.dim_date_v
    UNION ALL
    SELECT 'fact_headcount_v',
           COUNT(*) FROM people_analytics.semantic.fact_headcount_v
    UNION ALL
    SELECT 'fact_attrition_v',
           COUNT(*) FROM people_analytics.semantic.fact_attrition_v
    UNION ALL
    SELECT 'fact_compensation_v',
           COUNT(*) FROM people_analytics.semantic.fact_compensation_v
    UNION ALL
    SELECT 'fact_performance_v',
           COUNT(*) FROM people_analytics.semantic.fact_performance_v
    UNION ALL
    SELECT 'v_headcount_trend_v',
           COUNT(*) FROM people_analytics.semantic.v_headcount_trend_v
    UNION ALL
    SELECT 'v_attrition_trend_v',
           COUNT(*) FROM people_analytics.semantic.v_attrition_trend_v
    UNION ALL
    SELECT 'v_compensation_trend_v',
           COUNT(*) FROM people_analytics.semantic.v_compensation_trend_v
    UNION ALL
    SELECT 'dim_month_v',
           COUNT(*) FROM people_analytics.semantic.dim_month_v
) ORDER BY view_name;



In [0]:
-- =============================================================================
-- CELL 19 Change resilience test
-- Demonstrates the semantic layer absorbs source changes
-- =============================================================================

-- Simulate what happens if data engineering renames a column:
-- The semantic view uses an alias — Power BI column name never changes
-- Example: if silver.dim_employee renames 'business_unit' to 'org_unit'
-- we update the view here: SELECT org_unit AS business_unit
-- Power BI sees 'business_unit' unchanged — zero dashboard impact

SELECT
    'Change resilience demonstration' AS test,
    COUNT(DISTINCT business_unit)     AS distinct_business_units,
    COUNT(DISTINCT dept_name)         AS distinct_departments,
    COUNT(*)                          AS total_active_employees
FROM people_analytics.semantic.dim_employee_v
WHERE is_current = 1
  AND status     = 'Active';

SELECT 'Semantic layer build complete. Connect Power BI to semantic.* views.' AS status;